# 01 — Universe Inspection

Inspect the generated bond and client universes.
I want to verify that:
- Bond features and latent factors are sensible
- The similarity matrix has the right block structure
- Client archetypes are distinct in embedding space
- The affinity matrix A[k,n] varies meaningfully


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rfq_sim.core.config import SimConfig
from rfq_sim.core.bonds import BondUniverse
from rfq_sim.core.clients import ClientUniverse
from rfq_sim.utils.io import load_all

# Load pre-generated data -- run scripts/run_simulation.py first
data = load_all('../data')
obs         = data['observable']
bond_meta   = data['bond_metadata']
client_meta = data['client_metadata']
bond_gt     = data['bond_gt']
client_gt   = data['client_gt']
sim_matrix  = data['similarity']
affinity    = data['affinity']

print(f"Bonds: {len(bond_meta)}, Clients: {len(client_meta)}")
print(f"Similarity matrix shape: {sim_matrix.shape}")
print(f"Affinity matrix shape: {affinity.shape}")


## Bond Universe

In [ ]:
# Distribution of bonds across sectors, ratings, durations
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

bond_meta['sector'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Bonds by Sector'); axes[0].set_xlabel('')

bond_meta['rating'].value_counts().plot(kind='bar', ax=axes[1], color='salmon')
axes[1].set_title('Bonds by Rating')

bond_meta['liquidity_tier'].value_counts().sort_index().plot(
    kind='bar', ax=axes[2], color='seagreen'
)
axes[2].set_title('Bonds by Liquidity Tier')

plt.tight_layout()
plt.savefig('../data/plots_bond_universe.png', dpi=120, bbox_inches='tight')
plt.show()
print("Bond universe distributions plotted.")


In [ ]:
# Similarity matrix heatmap -- should show block structure by sector
# Sort bonds by sector to make the blocks visible
bond_order = bond_meta.sort_values(['sector', 'rating', 'duration_bucket']).index

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    sim_matrix[np.ix_(bond_order, bond_order)],
    ax=ax, cmap='viridis', vmin=0, vmax=1,
    xticklabels=False, yticklabels=False,
)
ax.set_title('Bond Similarity Matrix (sorted by sector/rating/duration)')
plt.tight_layout()
plt.savefig('../data/plots_similarity_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
# The sector blocks should be clearly visible as darker squares on the diagonal


In [ ]:
# Bond latent factors -- v_n should cluster by sector
from sklearn.decomposition import PCA

V = bond_gt[[c for c in bond_gt.columns if c.startswith('v_n_')]].values
pca = PCA(n_components=2)
V_2d = pca.fit_transform(V)

fig, ax = plt.subplots(figsize=(8, 6))
sectors = bond_meta['sector'].values
for sector in bond_meta['sector'].unique():
    mask = sectors == sector
    ax.scatter(V_2d[mask, 0], V_2d[mask, 1], label=sector, alpha=0.7, s=40)

ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('Bond Latent Factors v_n (PCA projection)')
ax.legend()
plt.tight_layout()
plt.show()
# Bonds from the same sector should cluster -- this is the ground truth
# structure that a recommender needs to recover from interaction data


## Client Universe

In [ ]:
# Client archetype distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

client_gt['archetype'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Client Archetype Distribution')

# rho_k distribution by archetype -- informed clients have higher rho
client_gt.boxplot(column='rho_k', by='archetype', ax=axes[1])
axes[1].set_title('Informativeness rho_k by Archetype')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()
# Sector specialists and liquidity seekers should have higher median rho_k


In [ ]:
# Client latent factors -- u_k should cluster by archetype
U = client_gt[[c for c in client_gt.columns if c.startswith('u_k_')]].values
U_2d = PCA(n_components=2).fit_transform(U)

archetypes = client_gt['archetype'].values
fig, ax = plt.subplots(figsize=(8, 6))
for arch in client_gt['archetype'].unique():
    mask = archetypes == arch
    ax.scatter(U_2d[mask, 0], U_2d[mask, 1], label=arch, alpha=0.7, s=40)

ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('Client Latent Factors u_k (PCA projection)')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Affinity matrix -- A[k, n] = u_k . v_n
# Should show that each client archetype has high affinity for specific bond groups

fig, ax = plt.subplots(figsize=(12, 8))
# Sort clients by archetype, bonds by sector
client_order = client_gt.sort_values('archetype').index
bond_order   = bond_meta.sort_values('sector').index

sns.heatmap(
    affinity[np.ix_(client_order, bond_order)],
    ax=ax, cmap='RdYlGn', center=0,
    xticklabels=False, yticklabels=False,
)
ax.set_title('Affinity Matrix A[k,n] = u_k . v_n\n(clients sorted by archetype, bonds by sector)')
ax.set_xlabel('Bonds (sorted by sector)')
ax.set_ylabel('Clients (sorted by archetype)')
plt.tight_layout()
plt.show()
# The block-diagonal structure is what a collaborative filtering model should recover
